In [1]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import multivariate_normal
from mpl_toolkits.mplot3d import Axes3D
import random
from random import random
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn import metrics
from sklearn.metrics import adjusted_rand_score
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt


In [2]:
from sklearn.datasets import fetch_kddcup99
# Load as pandas DataFrame
data = fetch_kddcup99(as_frame=True)
X = data.data
y = data.target

In [3]:
X.shape

(494021, 41)

In [4]:
print(np.unique(y))
print(len(np.unique(y)))

[b'back.' b'buffer_overflow.' b'ftp_write.' b'guess_passwd.' b'imap.'
 b'ipsweep.' b'land.' b'loadmodule.' b'multihop.' b'neptune.' b'nmap.'
 b'normal.' b'perl.' b'phf.' b'pod.' b'portsweep.' b'rootkit.' b'satan.'
 b'smurf.' b'spy.' b'teardrop.' b'warezclient.' b'warezmaster.']
23


In [5]:
X.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
0,0,b'tcp',b'http',b'SF',181,5450,0,0,0,0,...,9,9,1.0,0.0,0.11,0.0,0.0,0.0,0.0,0.0
1,0,b'tcp',b'http',b'SF',239,486,0,0,0,0,...,19,19,1.0,0.0,0.05,0.0,0.0,0.0,0.0,0.0
2,0,b'tcp',b'http',b'SF',235,1337,0,0,0,0,...,29,29,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0
3,0,b'tcp',b'http',b'SF',219,1337,0,0,0,0,...,39,39,1.0,0.0,0.03,0.0,0.0,0.0,0.0,0.0
4,0,b'tcp',b'http',b'SF',217,2032,0,0,0,0,...,49,49,1.0,0.0,0.02,0.0,0.0,0.0,0.0,0.0


In [6]:
continuous_features = [
    "duration", "src_bytes", "dst_bytes", "wrong_fragment", "urgent", "hot",
    "num_failed_logins", "num_compromised", "root_shell", "su_attempted", "num_root",
    "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds",
    "count", "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate"
]

In [7]:
X[continuous_features] = X[continuous_features].apply(
    pd.to_numeric)

/tmp/ipykernel_65454/2154176689.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[continuous_features] = X[continuous_features].apply(


In [8]:
X1 = X[continuous_features]

print(X1.shape)
print(X1.head())
print(X1.tail())
print(X1.dtypes)

(494021, 34)
   duration  src_bytes  dst_bytes  wrong_fragment  urgent  hot  \
0         0        181       5450               0       0    0   
1         0        239        486               0       0    0   
2         0        235       1337               0       0    0   
3         0        219       1337               0       0    0   
4         0        217       2032               0       0    0   

   num_failed_logins  num_compromised  root_shell  su_attempted  ...  \
0                  0                0           0             0  ...   
1                  0                0           0             0  ...   
2                  0                0           0             0  ...   
3                  0                0           0             0  ...   
4                  0                0           0             0  ...   

   dst_host_count  dst_host_srv_count  dst_host_same_srv_rate  \
0               9                   9                     1.0   
1              19          

In [9]:
percentages = y.value_counts(normalize=True) * 100  # multiply by 100 to get %
print(percentages)

labels
b'smurf.'              56.837665
b'neptune.'            21.699685
b'normal.'             19.691066
b'back.'                0.445932
b'satan.'               0.321646
b'ipsweep.'             0.252418
b'portsweep.'           0.210517
b'warezclient.'         0.206469
b'teardrop.'            0.198170
b'pod.'                 0.053439
b'nmap.'                0.046759
b'guess_passwd.'        0.010728
b'buffer_overflow.'     0.006073
b'land.'                0.004251
b'warezmaster.'         0.004048
b'imap.'                0.002429
b'rootkit.'             0.002024
b'loadmodule.'          0.001822
b'ftp_write.'           0.001619
b'multihop.'            0.001417
b'phf.'                 0.000810
b'perl.'                0.000607
b'spy.'                 0.000405
Name: proportion, dtype: float64


In [10]:
from sklearn.model_selection import train_test_split

X_sample, _, y_sample, _ = train_test_split(
    X1,
    y,
    train_size= 30000,
    stratify=y
)
X_sample = X_sample.reset_index(drop=True)
y_sample = y_sample.reset_index(drop=True)
print(y_sample.value_counts(normalize=True))


labels
b'smurf.'              0.568367
b'neptune.'            0.217000
b'normal.'             0.196900
b'back.'               0.004467
b'satan.'              0.003233
b'ipsweep.'            0.002533
b'portsweep.'          0.002100
b'warezclient.'        0.002067
b'teardrop.'           0.001967
b'pod.'                0.000533
b'nmap.'               0.000467
b'guess_passwd.'       0.000100
b'buffer_overflow.'    0.000067
b'imap.'               0.000033
b'warezmaster.'        0.000033
b'land.'               0.000033
b'rootkit.'            0.000033
b'loadmodule.'         0.000033
b'ftp_write.'          0.000033
Name: proportion, dtype: float64


In [11]:
y1 = np.zeros(len(y_sample))
              
for i in range(len(y_sample)):
    if y_sample[i] == b'smurf.':
        y1[i] = 0
    elif y_sample[i] == b'neptune.':
        y1[i] = 1
    elif y_sample[i] == b'normal.':
        y1[i] = 2
    else:
        y1[i] = -1

In [12]:
print(X_sample.shape)
y1.shape

(30000, 34)


(30000,)

In [13]:
 X_sample = X_sample.to_numpy(dtype=np.float64)

In [14]:
from sklearn.preprocessing import RobustScaler
X_scaled = RobustScaler().fit_transform(X_sample)

In [15]:
#PAM
!pip install scikit-learn-extra

from sklearn_extra.cluster import KMedoids

def pam(X_data, y_data, k, max_iter= 100):
    #start_time = time.time()
    kmedoids = KMedoids(n_clusters = k, metric = "manhattan", method = "pam",init="random", max_iter = 500)
    #end_time = time.time()
    kmedoids.fit(X_data)
    labels = kmedoids.labels_
    
    #diff_time = end_time-start_time
    non_outlier_indices = np.where(y_data != -1)[0]
    #est_clust_wo_outliers = labels[non_outlier_indices]
    ari = metrics.adjusted_rand_score(labels[non_outlier_indices], y_data[non_outlier_indices])#return labels, medoids
    return ari

Defaulting to user installation because normal site-packages is not writeable


In [ ]:
max_ari_pam = -1
for it in range(30):
    ari_curr= pam(X_data=X_scaled, y_data=y1, k=3)
    print(ari_curr)
    if ari_curr > max_ari_pam:
        max_ari_pam = ari_curr

0.050202690798596905
0.050202690798596905
0.050202690798596905
0.050202690798596905
0.050202690798596905
0.050202690798596905
0.050202690798596905
0.050202690798596905
0.050202690798596905
0.050202690798596905
0.050202690798596905


In [37]:
#euclidean distance
def euclid_distance(p1, p2):
    return np.sqrt(np.sum((p1-p2)**2))

#functions for cluster assignment and cluster mean update
def assign_clusters(X_data, clusters):
    n = X_data.shape[0]
    for k in clusters:
        clusters[k]["points"] = []  # Clear points each iteration (moved here)

    for i in range(n):
        curr_X = X_data[i, :]
        dist = [euclid_distance(curr_X, clusters[k]["center"]) for k in clusters]
        curr_cluster = np.argmin(dist)
        clusters[curr_cluster]["points"].append(i)  # Store index instead of point
    return clusters

def update_clusters(X_data, clusters):
    for k in clusters:
        indices = clusters[k]["points"]
        if indices:  # Check non-empty
            cluster_members = X_data[indices]  # Get points by index
            updated_center = cluster_members.mean(axis=0)
            clusters[k]["center"] = updated_center
    return clusters

In [38]:
def initialization_heur(X_data):

    n, d = X_data.shape
    indices = np.random.choice(n, 3, replace=False)
    centroids = X_data[indices, :]
    c1 =  X_data[indices, :]

    clusters = {0: {"center":c1[0,:], "points":[]},
                1: {"center":c1[1,:], "points":[]},
                2: {"center":c1[2,:], "points":[]}
                }

    clusters = assign_clusters(X_data,clusters)
    clusters = update_clusters(X_data,clusters)
    cov_initial1 = np.zeros((d, d))

    for i in clusters[0]["points"]:
        diff =  (X_data[i] - clusters[0]["center"]).reshape((d, 1))
        cov_initial1+= (diff @ diff.T)/ len(clusters[0]["points"])

    cov_initial2 = np.zeros((d, d))
    for i in clusters[1]["points"]:
        diff =  (X_data[i] - clusters[1]["center"]).reshape((d, 1))
        cov_initial2+= (diff @ diff.T)/ len(clusters[1]["points"])

    cov_initial3 = np.zeros((d, d))
    for i in clusters[2]["points"]:
        diff =  (X_data[i] - clusters[2]["center"]).reshape((d, 1))
        cov_initial3+= (diff @ diff.T)/ len(clusters[2]["points"])
    covar = [cov_initial1, cov_initial2, cov_initial3]
    return centroids, covar

In [39]:
def heuristic_clustering(X_data, y_data, K=3, max_outer_iter=1, max_inner_iter= 500, tol=1e-4):

    n, d = X_data.shape
    for _ in range(max_outer_iter):

        # Initialization
        centroids, cov = initialization_heur(X_data)
        centroids = centroids.astype(np.float64)
        Omega = [np.linalg.inv(c.astype(np.float64)+1e-6*np.eye(d)) for c in cov]

        prev_centroids = centroids.copy()
        est_clust_assignments = np.zeros(n, dtype=int)

        for m in range(max_inner_iter):

            #  Assignment
            for i in range(n):
                dist = []

                for j in range(K):
                    const_term = ((1 / 2) ** d) * (1 / np.pi ** (d / 2)) \
                                 * (np.linalg.det(Omega[j]) ** 0.5) * (1 / n)

                    exp_term = (2 ** (1 - d / 2)) * (np.linalg.det(Omega[j]) ** 0.5) \
                               / (n * (np.pi ** (d / 2)))

                    diff = (X_data[i] - centroids[j]).reshape((d, 1))
                    dist_val = const_term - exp_term * np.exp(
                        -0.5 * (diff.T @ Omega[j] @ diff)
                    )
                    dist.append(dist_val)

                est_clust_assignments[i] = np.argmin(dist)

            # Centroid & Covariance Update 
            for c in range(K):
                indices = np.where(est_clust_assignments == c)[0]
                if len(indices) == 0:
                    continue

                # differences
                diffs = [(X_data[e] - centroids[c]).reshape((d, 1)) for e in indices]

                # log-domain exponents
                log_exponents = np.array([-0.5 * (diff.T @ Omega[c] @ diff).item() for diff in diffs], dtype=np.float64)

                # log-sum-exp trick for safe sum of weights
                max_log = np.max(log_exponents)
                sum_weights = np.sum(np.exp(log_exponents - max_log)) * np.exp(max_log)

                # compute exact original weights safely
                weights = np.exp(log_exponents)

                # Centroid update
                weighted_sum = np.zeros(d, dtype=np.float64)
                for w, diff, e in zip(weights, diffs, indices):
                    weighted_sum += w * Omega[c].dot(X_data[e])
                centroids[c] = np.linalg.inv(Omega[c]) @ (weighted_sum / sum_weights)
                
                
                # differences
                diffs = [(X_data[e] - centroids[c]).reshape((d, 1)) for e in indices]

                # log-domain exponents
                log_exponents = np.array([-0.5 * (diff.T @ Omega[c] @ diff).item() for diff in diffs], dtype=np.float64)

                # log-sum-exp trick for safe sum of weights
                max_log = np.max(log_exponents)
                sum_weights = np.sum(np.exp(log_exponents - max_log)) * np.exp(max_log)

                # compute exact original weights safely
                weights = np.exp(log_exponents)
                # Covariance update
                weighted_cov = np.zeros((d, d), dtype=np.float64)
                for w, diff in zip(weights, diffs):
                    weighted_cov += w * (diff @ diff.T)
                weighted_cov /= sum_weights
                weighted_cov += 1e-6 * np.eye(d)  # minimal regularization

                # Omega update
                Omega[c] = (np.trace(Omega[c] @ weighted_cov) / d) * np.linalg.pinv(weighted_cov)

            # Check convergence
            diff_centroids = np.linalg.norm(prev_centroids - centroids, axis=1)
            if np.all(diff_centroids < tol):
                break
            prev_centroids = centroids.copy()
     
        #  Evaluation
        non_outlier_indices = np.where(y_data != -1)[0]
        est_clust_wo_outliers = est_clust_assignments[non_outlier_indices]
        ari = metrics.adjusted_rand_score(est_clust_assignments[non_outlier_indices], y_data[non_outlier_indices ])
        return ari,centroids, np.linalg.inv(Omega)# est_clust_assignments, c

In [40]:
max_ari_mm = -1
best_m = 0
best_cov = 0
for it in range(30):
    ari_curr, m, cov = heuristic_clustering(X_scaled, y1)
    print(ari_curr)
    if ari_curr > max_ari_mm:
        max_ari_mm = ari_curr
        best_m = m
        best_cov = cov

0.6555030062303052
0.668570947620186
0.579812607755522
0.5700606591177639
0.4654306729356153
0.5777023420517312
0.5517491264210675
0.5518174664268578
0.5836204300154796
0.7724881531590833
0.568023352936421
0.4788379537688277
0.6718139565182139
0.45950235095812425
0.5518174664268578
0.5431759486514053
0.48209277202500345
0.4052805269636211
0.4353290718432741
0.5633781525515346
0.5518174664268578
0.39123354239144076
0.4067821623487669
0.5517491264210675
0.029012087977430576
0.7278934227924838
0.6665876518575345
0.5431183976926646
0.5518174664268578
0.5431909060167435


In [41]:
from sklearn import metrics
!pip install studenttmixture
from studenttmixture import EMStudentMixture
max_ari_t = -1
best_model_t = None
best_labels_t = None

for it in range(30):
    try:
        model = EMStudentMixture(
            n_components=3,
            max_iter=500,
            tol=1e-4,
            verbose=False
        )

        model.fit(X_scaled)

        # Check whether model converged
        if model.df_ is None:
            ari_curr = -1
        else:
            labs = model.predict(X_scaled)

            # Exclude observations where y1 == -1
            mask = (y1 != -1)

            ari_curr = metrics.adjusted_rand_score(
                y1[mask],
                labs[mask]
            )
            print(ari_curr)

        # Keep the best result
        if ari_curr > max_ari_t:
            max_ari_t = ari_curr
            best_model_t = model
            best_labels_t = labs

        print(f"Iteration {it}: ARI = {ari_curr:.4f}")

    except Exception as e:
        print(f"Iteration {it}: EM failed -> {e}")

print(f"\nBest ARI: {max_ari_t:.4f}")


Defaulting to user installation because normal site-packages is not writeable
0.5092933174423608
Iteration 0: ARI = 0.5093
0.5092933174423608
Iteration 1: ARI = 0.5093
0.5092933174423608
Iteration 2: ARI = 0.5093
0.5092933174423608
Iteration 3: ARI = 0.5093
0.5092933174423608
Iteration 4: ARI = 0.5093
0.5092933174423608
Iteration 5: ARI = 0.5093
0.5092933174423608
Iteration 6: ARI = 0.5093
0.5092933174423608
Iteration 7: ARI = 0.5093
0.5092933174423608
Iteration 8: ARI = 0.5093
0.5092933174423608
Iteration 9: ARI = 0.5093
0.5092933174423608
Iteration 10: ARI = 0.5093
0.5092933174423608
Iteration 11: ARI = 0.5093
0.5092933174423608
Iteration 12: ARI = 0.5093
0.5092933174423608
Iteration 13: ARI = 0.5093
0.5092933174423608
Iteration 14: ARI = 0.5093
0.5092933174423608
Iteration 15: ARI = 0.5093
0.5092933174423608
Iteration 16: ARI = 0.5093
0.5092933174423608
Iteration 17: ARI = 0.5093
0.5092933174423608
Iteration 18: ARI = 0.5093
0.5092933174423608
Iteration 19: ARI = 0.5093
0.5092933174

In [42]:
#trimmed k means

In [43]:
import numpy as np
from itertools import permutations
from sklearn.metrics import adjusted_rand_score

def trimmed_kmeans(data, k, num_outliers, max_iter=100, n_init=5,
                   tol=1e-4, random_state=None):
    """
    Trimmed K-means.

    Parameters
    ----------
    data : ndarray
        n x d data matrix.

    k : int
        Number of clusters.

    num_outliers : int
        Number of observations to trim.

    max_iter : int
        Maximum iterations per initialization.

    n_init : int
        Number of random initializations.

    tol : float
        Convergence tolerance.

    random_state : int
        Random seed.

    Returns
    -------
    inliers : ndarray
        Indices of retained observations.

    labels : ndarray
        Cluster labels for the retained observations.

    outliers : ndarray
        Indices of trimmed observations.

    centers : ndarray
        Final cluster centers.
    """

    n = len(data)

    if num_outliers >= n:
        raise ValueError("num_outliers must be smaller than n")

    if k >= n - num_outliers:
        raise ValueError("k must be smaller than the number of inliers")

    rng = np.random.default_rng(random_state)

    n_inliers = n - num_outliers

    best_cost = np.inf
    best_inliers = None
    best_labels = None
    best_centers = None

    for init in range(n_init):

        # Random initialization
        center_indices = rng.choice(n, size=k, replace=False)
        centers = data[center_indices].copy()

        previous_inliers = None

        for iteration in range(max_iter):
            # Assign every point to its nearest center
            distances = np.sum((data[:, None, :] - centers[None, :, :]) ** 2,axis=2)
            labels_all = np.argmin(distances, axis=1)

            # Distance to assigned center
            min_distances = distances[np.arange(n),labels_all]
            
            # Trim the furthest observations
            inliers = np.argpartition(min_distances,n_inliers - 1)[:n_inliers]
            inliers = np.sort(inliers)
            # Check whether trimming has converged
            if (previous_inliers is not None and np.array_equal(inliers, previous_inliers)):
                break

            previous_inliers = inliers.copy()
            # Recalculate cluster centers using inliers only
            new_centers = np.zeros_like(centers)
            for cluster in range(k):
                cluster_indices = inliers[labels_all[inliers] == cluster]
                if len(cluster_indices) == 0:
                    # Reinitialize an empty cluster
                    new_centers[cluster] = data[rng.choice(inliers)]
                else:
                    new_centers[cluster] = np.mean(data[cluster_indices],axis=0)
            # Check center convergence
            center_shift = np.linalg.norm(new_centers - centers)
            centers = new_centers
            if center_shift < tol:
                break

        # Calculate final assignments
        distances = np.sum((data[:, None, :] - centers[None, :, :]) ** 2,axis=2)
        labels_all = np.argmin(distances, axis=1)
        min_distances = distances[np.arange(n),labels_all]
    
        # Trim again using final centers
        inliers = np.argpartition(min_distances,n_inliers - 1)[:n_inliers]
        inliers = np.sort(inliers)

        # Trimmed K-means objective
        cost = np.sum(min_distances[inliers])
        # Keep the best initialization
        if cost < best_cost:
            best_cost = cost
            best_inliers = inliers.copy()
            best_labels = labels_all[inliers].copy()
            best_centers = centers.copy()

    # Everything not retained is an outlier
    outliers = np.setdiff1d(np.arange(n),best_inliers)
    return best_inliers, best_labels, outliers, best_centers

In [44]:
def trimmed_kmeans_ari(data,k,num_outliers,true_labels,max_iter=100,n_init=5,tol=1e-4,random_state=None):
    # Run trimmed K-means
    inliers, labels, outliers, centers = trimmed_kmeans(data=data, k=k, num_outliers=num_outliers,
        max_iter=max_iter, n_init=n_init, tol=tol, random_state=random_state)

    # Put the labels back into the original data ordering
    n = len(data)
    full_labels = np.full(n, -1)
    full_labels[inliers] = labels
    # 
    valid = (true_labels != -1) & (full_labels != -1)

    y_true_valid = true_labels[valid].astype(int)
    y_pred_valid = full_labels[valid].astype(int)

    best_score = -np.inf
    best_perm = None

    for perm in permutations(range(k)):
        perm = np.array(perm)
        score = np.mean(perm[y_true_valid] == y_pred_valid)
        if score > best_score:
            best_score = score
            best_perm = perm

    aligned_true = true_labels.astype(int).copy()
    mask = aligned_true != -1
    aligned_true[mask] = best_perm[aligned_true[mask]]
    rng = np.random.default_rng(0)
    final_pred = full_labels.copy()
    # Assign trimmed true-inliers randomly
    trimmed = ((final_pred == -1)& (aligned_true != -1))
    for i in np.where(trimmed)[0]:
        true_k = aligned_true[i]
        choices = np.delete(np.arange(k),true_k)
        final_pred[i] = rng.choice(choices)

    #ARI
    valid_ind = aligned_true != -1
    ari = adjusted_rand_score(aligned_true[valid_ind],final_pred[valid_ind])
    return ari

In [45]:
max_ari_trimmed = -1

for it in range(20):

    ari_curr = trimmed_kmeans_ari(X_scaled, k=3, num_outliers=np.sum(y1 == -1), true_labels=y1,
        max_iter=500, n_init=50, random_state=it)
    print(ari_curr)
    if ari_curr > max_ari_trimmed:
        max_ari_trimmed = ari_curr

print("Best ARI:", max_ari_trimmed)
    

0.11258037522539224
0.11578161574318642
0.11258037522539224
0.11578161574318642
0.11258037522539224
0.11578161574318642
0.11258037522539224
0.11258037522539224
0.11258037522539224
0.11578161574318642
0.11578161574318642
0.11578161574318642
0.11258037522539224
0.11258037522539224
0.11258037522539224
0.1130705370588237
0.11578161574318642
0.11627452651786714
0.11627452651786714
0.11578161574318642
Best ARI: 0.11627452651786714
